**Copyright 2026 The MediaPipe Authors. All Rights Reserved.**

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Semantic Retrieval with MediaPipe Tasks

This notebook shows you how to use the MediaPipe Tasks Python API (`SemanticRetriever` and `UniversalEmbedder`) to index text documents, images, and multimodal content into a vector store and retrieve the most relevant records using natural language or multimodal queries.

## Preparation

Let's start with installing MediaPipe.

In [ ]:
!pip install -q mediapipe

After installing your dependencies, download an EmbeddingGemma 2 (LiteRT-LM) universal embedder model for this example:

* **Text (270M)**: `https://huggingface.co/litert-community/embeddinggemma-2-text-270m-litert-lm/resolve/main/embeddinggemma-2-text-270m.litertlm`
* **Text + Vision (440M)**: `https://huggingface.co/litert-community/embeddinggemma-2-text-vision-440m-litert-lm/resolve/main/embeddinggemma-2-text-vision-440m.litertlm`
* **Multimodal (740M)**: `https://huggingface.co/litert-community/embeddinggemma-2-740m-litert-lm/resolve/main/embeddinggemma-2-740m.litertlm`

Below we download the Text + Vision (440M) model (`embeddinggemma-2-text-vision-440m.litertlm`).

In [ ]:
#@title Start downloading here.
!wget -O embeddinggemma-2-text-vision-440m.litertlm -q https://huggingface.co/litert-community/embeddinggemma-2-text-vision-440m-litert-lm/resolve/main/embeddinggemma-2-text-vision-440m.litertlm

Next, download sample images (`burger.jpg` and `cat.jpg`) to index alongside text documents in the vector store.

In [ ]:
import urllib.request

IMAGE_FILENAMES = ['burger.jpg', 'cat.jpg']

for name in IMAGE_FILENAMES:
  url = f'https://storage.googleapis.com/mediapipe-tasks/image_classifier/{name}'
  urllib.request.urlretrieve(url, name)

## Indexing and retrieving documents and images

To use `SemanticRetriever`, first create a `UniversalEmbedder` with the downloaded model and pass it into `SemanticRetrieverOptions`. You can then insert text documents (`insert_document`), images (`insert_image`), or multimodal content (`insert_content`) with optional key-value metadata, and query the index with `retrieve`.

In [ ]:
from mediapipe.tasks import python
from mediapipe.tasks.python.retrieval import semantic_retriever
from mediapipe.tasks.python.retrieval import universal_embedder

# Initialize the UniversalEmbedder
base_options = python.BaseOptions(
    model_asset_path='embeddinggemma-2-text-vision-440m.litertlm'
)
embedder_options = universal_embedder.UniversalEmbedderOptions(
    base_options=base_options,
    l2_normalize=True,
)

with universal_embedder.UniversalEmbedder.create_from_options(
    embedder_options
) as embedder:
  retriever_options = semantic_retriever.SemanticRetrieverOptions(
      embedder=embedder,
      embedding_dimension=768,
      chunk_size=512,
      chunk_overlap=100,
      chunking_mode=semantic_retriever.ChunkingMode.CHARACTER,
  )

  with semantic_retriever.SemanticRetriever.create_from_options(
      retriever_options
  ) as retriever:
    # Insert text documents with metadata
    retriever.insert_document(
        doc_id='doc_burger',
        text='A grilled beef burger topped with melted cheddar, lettuce, and tomato on a toasted sesame bun.',
        metadata={'category': 'food', 'type': 'text'},
    )
    retriever.insert_document(
        doc_id='doc_cat',
        text='Domestic cats are playful, curious companions known for purring and napping in sunny spots.',
        metadata={'category': 'animals', 'type': 'text'},
    )

    # Insert images with metadata
    retriever.insert_image(
        image_id='img_burger',
        image_path='burger.jpg',
        metadata={'category': 'food', 'type': 'image'},
    )
    retriever.insert_image(
        image_id='img_cat',
        image_path='cat.jpg',
        metadata={'category': 'animals', 'type': 'image'},
    )

    print('Indexed record IDs:', retriever.get_all_record_ids())

    # Retrieve nearest records for a natural-language query
    query = 'feline pet'  #@param {type:"string"}
    limit = 4  #@param {type:"integer"}
    min_similarity = 0.0  #@param {type:"number"}

    result = retriever.retrieve(
        query=query,
        limit=limit,
        min_similarity=min_similarity,
    )

    print(f'\nTop matches for "{query}":')
    for record in result.records:
      print(
          f'  [{record.score:.4f}] id={record.id} '
          f'metadata={record.metadata} text={record.text!r}'
      )

## Multimodal content, metadata filtering, and record management

You can also index multi-part records via `insert_content` using `TaskPartKind`, filter search results by metadata (`metadata_filter`), query using image parts, and delete specific or filtered records (`delete_record`, `delete_with_metadata_filter`, `delete_all`).

In [ ]:
with open('cat.jpg', 'rb') as f:
  cat_bytes = f.read()

with universal_embedder.UniversalEmbedder.create_from_options(
    embedder_options
) as embedder:
  retriever_options = semantic_retriever.SemanticRetrieverOptions(
      embedder=embedder
  )

  with semantic_retriever.SemanticRetriever.create_from_options(
      retriever_options
  ) as retriever:
    # Insert records with metadata for filtering
    retriever.insert_document(
        doc_id='doc_burger',
        text='Chef special cheeseburger combo meal',
        metadata={'category': 'food'},
    )
    retriever.insert_document(
        doc_id='doc_cat',
        text='Tabby kitten portrait',
        metadata={'category': 'animals'},
    )

    # Insert multimodal content combining text and image parts
    retriever.insert_content(
        record_id='combo_burger',
        parts=[
            {
                'kind': semantic_retriever.TaskPartKind.TEXT,
                'text': 'Juicy beef burger with fries',
            },
            {
                'kind': semantic_retriever.TaskPartKind.IMAGE,
                'file_path': 'burger.jpg',
            },
        ],
    )

    # Retrieve with a metadata filter
    filtered_result = retriever.retrieve(
        query='something delicious to eat',
        limit=5,
        min_similarity=0.0,
        metadata_filter={'category': 'food'},
    )
    print('Filtered results (category="food"):')
    for record in filtered_result.records:
      print(f'  [{record.score:.4f}] id={record.id} metadata={record.metadata}')

    # Retrieve using an image query part
    image_query_result = retriever.retrieve(
        query=[
            {
                'kind': semantic_retriever.TaskPartKind.IMAGE,
                'image_bytes': cat_bytes,
            }
        ],
        limit=5,
        min_similarity=0.0,
    )
    print('\nResults for image query (cat.jpg):')
    for record in image_query_result.records:
      print(f'  [{record.score:.4f}] id={record.id} metadata={record.metadata}')

    # Delete records by metadata filter and inspect remaining IDs
    retriever.delete_with_metadata_filter({'category': 'food'})
    print(
        '\nRemaining record IDs after deleting category="food":',
        retriever.get_all_record_ids(),
    )